# 3-2 アンケートで顧客の本音を読む / Reading Customer Insights from Survey Data

『AIに頼んで動かす Python実務データ分析』第3章2節の参照用ノートブックです。
Reference notebook for Chapter 3, Section 2 of *Data Analysis with AI and Python*.

**使い方 / How to use**
1. 最初のセルの `LANG` を `"ja"`（日本語）または `"en"`（英語）にします。 / Set `LANG` in the first cell to `"ja"` or `"en"`.
2. メニューの「ランタイム → すべてのセルを実行」で、上から順に実行します。 / Run all cells from top to bottom (Runtime → Run all).

本文では、AIへのプロンプトからコードを作っていきます。このノートブックは、うまく動かないときに答え合わせをするためのものです。
In the book, you build this code by prompting an AI. Use this notebook to check your results when something doesn't work.

In [ ]:
# ===== 設定 / Settings =====
LANG = "ja"   # "ja" = 日本語 / "en" = English
SAVE_FIGURES = True   # True にすると figures/ja または figures/en に図を保存します / Save figures to figures/<LANG>

# 配布データの置き場所（サポートページのGitHubリポジトリ）/ Where the companion data is hosted
BASE_URL = "https://raw.githubusercontent.com/rekishi-data/ai-python-data-analysis/main/data/"

## 準備 / Setup

必要なライブラリと、日本語版ではグラフ用の日本語フォントをインストールします。
Installs the required libraries (and a Japanese font for the Japanese edition).

In [ ]:
import os, sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install("janome", "prince==0.21.0", "wordcloud")

JP_FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if LANG == "ja" and not os.path.exists(JP_FONT):
    subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True, check=True)
print("準備完了 / Setup complete")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# 図の共通設定（モノクロ・紙面サイズ） / Common figure settings (monochrome, print size)
if LANG == "ja":
    fm.fontManager.addfont(JP_FONT)
    plt.rcParams["font.family"] = fm.FontProperties(fname=JP_FONT).get_name()
plt.rcParams.update({
    "font.size": 8, "axes.titlesize": 9, "axes.edgecolor": "black",
    "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300,
})
FIG_W = 4.5  # 全幅の図の幅（インチ） / Full-width figure width (inches)

def save(fig, name):
    """図を figures/<LANG>/ に保存する / Save a figure to figures/<LANG>/"""
    if SAVE_FIGURES:
        os.makedirs(f"figures/{LANG}", exist_ok=True)
        fig.savefig(f"figures/{LANG}/{name}.png", bbox_inches="tight")

In [ ]:
# ===== 表示用のラベル（日本語／英語） / Display labels (Japanese / English) =====
QUESTIONS = [f"q{i:02d}" for i in range(1, 13)]
L = {
  "ja": {
    "q_short": ["価格", "セール", "コスパ", "トレンド", "SNS", "個性", "素材", "耐久性", "着心地", "環境", "倫理", "理念"],
    "q_long": ["価格が安い", "セール・割引", "値段に見合った価値", "最新のトレンド", "SNSで話題", "個性的なデザイン",
               "素材の品質", "耐久性", "着心地・サイズ感", "環境配慮の素材", "倫理的な生産", "理念への共感"],
    "ages": ["10代", "20代", "30代", "40代", "50代以上"],
    "factors": ["価格重視", "トレンド重視", "品質重視", "エシカル重視"],
    "images": {"affordable": "手頃な価格", "classic": "定番", "cute": "かわいい", "eco_friendly": "環境にやさしい",
               "functional": "機能的", "luxurious": "高級感", "trendy": "トレンド感", "unique": "個性的"},
    "factor_no": "因子の番号", "eigenvalue": "固有値", "eigen1": "固有値＝1",
    "mean_score": "因子得点の平均", "brand": "ブランド", "image": "イメージ",
    "axis1": "第1軸", "axis2": "第2軸", "count": "出現回数", "question": "質問",
  },
  "en": {
    "q_short": ["Price", "Sales", "Value", "Trends", "Social", "Unique", "Material", "Durable", "Comfort", "Eco", "Ethical", "Values"],
    "q_long": ["Low price", "Sales and discounts", "Value for money", "Latest trends", "Social media buzz", "Unique design",
               "Material quality", "Durability", "Comfort and fit", "Eco-friendly materials", "Ethical production", "Brand values"],
    "ages": ["Teens", "20s", "30s", "40s", "50+"],
    "factors": ["Price-conscious", "Trend-conscious", "Quality-conscious", "Ethics-conscious"],
    "images": {"affordable": "affordable", "classic": "classic", "cute": "cute", "eco_friendly": "eco-friendly",
               "functional": "functional", "luxurious": "luxurious", "trendy": "trendy", "unique": "unique"},
    "factor_no": "Factor number", "eigenvalue": "Eigenvalue", "eigen1": "Eigenvalue = 1",
    "mean_score": "Mean factor score", "brand": "Brand", "image": "Image",
    "axis1": "Dimension 1", "axis2": "Dimension 2", "count": "Frequency", "question": "Question",
  },
}[LANG]
AGE_ORDER = ["10s", "20s", "30s", "40s", "50+"]

## データの読み込み / Loading the data

2つのファイルがなければ、配布データの置き場所から自動でダウンロードします。
If the two files are not found, they are downloaded automatically from the companion data site.

In [ ]:
import urllib.request

for name in ["survey_responses.csv", "brand_image_responses.csv"]:
    if not os.path.exists(name):
        try:
            urllib.request.urlretrieve(BASE_URL + name, name)
            print("downloaded:", name)
        except Exception as e:
            print(f"{name} をダウンロードできませんでした。左のファイルパネルからアップロードしてください。")
            print(f"Could not download {name}. Please upload it from the Files panel on the left. ({e})")

df = pd.read_csv("survey_responses.csv")
bi = pd.read_csv("brand_image_responses.csv")
print(df.shape, bi.shape)
print(df.isna().sum()[df.isna().sum() > 0])
df.head()

## 3-2-1 まずデータを眺める / Taking a first look at the data

年代別に12の質問の平均値を比べます（図3-2-1）。 / Compare the mean of the 12 questions by age group (Figure 3-2-1).

In [ ]:
m = df.groupby("age_group")[QUESTIONS].mean().loc[AGE_ORDER]

fig, ax = plt.subplots(figsize=(FIG_W, 2.6))
im = ax.imshow(m.values, cmap="Greys", vmin=2, vmax=4.5, aspect="auto")
nr, nc = im.get_array().shape
ax.set_xticks(np.arange(-0.5, nc, 1), minor=True); ax.set_yticks(np.arange(-0.5, nr, 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.8); ax.tick_params(which="minor", length=0)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        v = m.values[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6.5, color="white" if v > 3.5 else "black")
ax.set_xticks(range(12), [f"q{j+1:02d} {L['q_short'][j]}" for j in range(12)],
              fontsize=6.5, rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(5), L["ages"])
for s in ax.spines.values(): s.set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.8).ax.tick_params(labelsize=6.5)
fig.tight_layout(); save(fig, "fig3-2-1_heatmap"); plt.show()
m.round(2)

## 3-2-2 因子分析 / Factor analysis

### データが因子分析に向いているか / Is the data suitable for factor analysis?

KMOとバートレットの球面性検定を計算します。ライブラリのバージョンの違いで動かなくなるのを避けるため、ここではNumPyとSciPyで直接計算しています。
We compute the KMO measure and Bartlett's test of sphericity directly with NumPy and SciPy, to avoid library version problems.

In [ ]:
from scipy import stats

X = df[QUESTIONS]
R = X.corr().values
n, p = X.shape

# KMO（Kaiser-Meyer-Olkin）
inv = np.linalg.inv(R)
partial = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv)))
off = ~np.eye(p, dtype=bool)
kmo = (R[off] ** 2).sum() / ((R[off] ** 2).sum() + (partial[off] ** 2).sum())

# バートレットの球面性検定 / Bartlett's test of sphericity
chi2 = -(n - 1 - (2 * p + 5) / 6) * np.log(np.linalg.det(R))
dof = p * (p - 1) / 2
p_value = stats.chi2.sf(chi2, dof)

print(f"KMO = {kmo:.2f}")
print(f"Bartlett: chi2 = {chi2:.1f}, df = {dof:.0f}, p = {p_value:.2e}")

### 因子の数を決める / Choosing the number of factors (Figure 3-2-2)

In [ ]:
eigen = np.linalg.eigvalsh(R)[::-1]
print("固有値 / Eigenvalues:", np.round(eigen, 2))

fig, ax = plt.subplots(figsize=(FIG_W, 2.2))
ax.plot(range(1, 13), eigen, "k-o", ms=4, mfc="white")
ax.axhline(1, color="black", ls="--", lw=0.8)
ax.text(12, 1.05, L["eigen1"], ha="right", fontsize=7)
ax.set_xticks(range(1, 13)); ax.set_xlabel(L["factor_no"]); ax.set_ylabel(L["eigenvalue"])
fig.tight_layout(); save(fig, "fig3-2-2_scree"); plt.show()

### 因子を取り出す / Extracting the factors

scikit-learn の `FactorAnalysis` でバリマックス回転をします。因子の順番と符号は計算のたびに変わることがあるので、本文の表と同じ並び（価格・トレンド・品質・エシカル）と符号にそろえています。
We use scikit-learn's `FactorAnalysis` with varimax rotation. Because the order and sign of factors can vary, we align them with the table in the book (price, trend, quality, ethics).

In [ ]:
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)
fa = FactorAnalysis(n_components=4, rotation="varimax", random_state=0).fit(Xs)
load = fa.components_.T          # 行 = 質問、列 = 因子 / rows = questions, columns = factors
scores = fa.transform(Xs)

# 因子の並びと符号をそろえる / Align factor order and sign
groups = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]   # 価格・トレンド・品質・エシカル
order, signs = [], []
for g in groups:
    k = int(np.argmax(np.abs(load[g]).sum(axis=0)))
    order.append(k); signs.append(np.sign(load[g, k].sum()))
load = load[:, order] * signs
scores = scores[:, order] * signs

loadings = pd.DataFrame(load, index=[f"{q} {t}" for q, t in zip(QUESTIONS, L["q_long"])], columns=L["factors"])
print("因子負荷量 / Factor loadings")
display(loadings.round(2).style.format("{:.2f}").apply(
    lambda col: ["font-weight: bold" if abs(v) >= 0.4 else "" for v in col]))
print("説明率 / Proportion of variance:", np.round((load ** 2).sum(axis=0) / 12, 3))

### 年代別の因子得点 / Factor scores by age group (Figure 3-2-3)

In [ ]:
F = pd.DataFrame(scores, columns=L["factors"])
g = F.groupby(df["age_group"]).mean().loc[AGE_ORDER]

fig, ax = plt.subplots(figsize=(FIG_W, 2.6))
x = np.arange(5); w = 0.2
styles = [("white", ""), ("#bbbbbb", ""), ("#555555", ""), ("white", "////")]
for k, (color, hatch) in enumerate(styles):
    ax.bar(x + (k - 1.5) * w, g.iloc[:, k], w, color=color, edgecolor="black", hatch=hatch, lw=0.6, label=g.columns[k])
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, L["ages"]); ax.set_ylabel(L["mean_score"])
ax.legend(ncol=4, fontsize=6.5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.15))
fig.tight_layout(); save(fig, "fig3-2-3_factor_scores"); plt.show()
g.round(2)

## 3-2-3 対応分析 / Correspondence analysis (Figure 3-2-4)

In [ ]:
import prince

ct = pd.crosstab(bi["image"], bi["brand"])[["Brand A", "Brand B", "Brand C", "Brand D", "Our Store"]]
display(ct.rename(index=L["images"]))

ca = prince.CA(n_components=2, random_state=0).fit(ct.T)   # 行 = ブランド / rows = brands
rows = ca.row_coordinates(ct.T)
cols = ca.column_coordinates(ct.T)

# 軸の向きを本文の図にそろえる（Brand D を右、Brand A を上） / Orient axes as in the book
sx = np.sign(rows.loc["Brand D", 0]); sy = np.sign(rows.loc["Brand A", 1])
rows[0] *= sx; cols[0] *= sx; rows[1] *= sy; cols[1] *= sy
pct = ca.percentage_of_variance_

fig, ax = plt.subplots(figsize=(FIG_W, 3.6))
ax.scatter(rows[0], rows[1], marker="s", s=40, color="black", label=L["brand"])
ax.scatter(cols[0], cols[1], marker="o", s=30, facecolor="white", edgecolor="black", label=L["image"])
# ラベルの位置（重なり防止） / Label offsets to avoid overlaps
off = {"Our Store": (6, -10, "left"), "cute": (-6, 2, "right"), "trendy": (-6, -6, "right"),
       "functional": (-6, -3, "right"), "eco_friendly": (-4, -11, "right"), "unique": (-6, -3, "right")}
for name, (a, b) in rows.iterrows():
    dx, dy, ha = off.get(name, (6, 4, "left"))
    ax.annotate(name, (a, b), xytext=(dx, dy), textcoords="offset points", fontsize=7.5, fontweight="bold", ha=ha)
for name, (a, b) in cols.iterrows():
    dx, dy, ha = off.get(name, (6, -3, "left"))
    ax.annotate(L["images"][name], (a, b), xytext=(dx, dy), textcoords="offset points", fontsize=7, ha=ha)
ax.axhline(0, color="gray", lw=0.6, ls=":"); ax.axvline(0, color="gray", lw=0.6, ls=":")
ax.set_xlim(-1.0, 1.25)
ax.set_xlabel(f"{L['axis1']}（{pct[0]:.0f}%）" if LANG == "ja" else f"{L['axis1']} ({pct[0]:.0f}%)")
ax.set_ylabel(f"{L['axis2']}（{pct[1]:.0f}%）" if LANG == "ja" else f"{L['axis2']} ({pct[1]:.0f}%)")
ax.legend(fontsize=7, frameon=False, loc="upper right")
fig.tight_layout(); save(fig, "fig3-2-4_ca"); plt.show()
print("累積説明率 / Cumulative:", round(pct[0] + pct[1], 1), "%")

## 3-2-4 テキストマイニング / Text mining

日本語版は janome で名詞を取り出し、英語版は単語に分けて一般的な語（ストップワード）を除きます。
The Japanese edition extracts nouns with janome; the English edition splits words and removes common stop words.

In [ ]:
from collections import Counter
import re

COMMENT = "comment_ja" if LANG == "ja" else "comment_en"
comments = df[COMMENT].fillna("")
print("回答数 / Comments:", (comments != "").sum())

if LANG == "ja":
    from janome.tokenizer import Tokenizer
    tok = Tokenizer()
    def words(text):
        return [t.base_form for t in tok.tokenize(text)
                if t.part_of_speech.split(",")[0] == "名詞"
                and t.part_of_speech.split(",")[1] in ("一般", "サ変接続", "固有名詞", "形容動詞語幹")]
    BASE_STOP = set()
    EXTRA_STOP = {"他", "型", "高め"}          # 図3-2-5を見て追加する除外語 / stop words added after Figure 3-2-5
else:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    BASE_STOP = set(ENGLISH_STOP_WORDS) | {"like", "bit", "nice", "d", "ll", "s", "t", "wish", "want", "happy", "love",
                                           "great", "lot", "little", "make", "use", "share", "add", "lower"}
    EXTRA_STOP = {"hard", "tell", "slightly", "better", "high", "hesitate", "buy", "easy"}   # 図3-2-5を見て追加 / added after Figure 3-2-5
    def words(text):
        ws = re.findall(r"[a-z]+", text.lower())
        # 簡単な原形化（複数形の s を取る） / Simple lemmatization (strip plural -s)
        return [w[:-1] if w.endswith("s") and len(w) > 4 and not w.endswith("ss") else w for w in ws]

STOP = BASE_STOP | EXTRA_STOP
counts_all = Counter(w for c in comments for w in words(c) if w not in BASE_STOP)   # 除外語の処理前 / before extra stop words
counts = Counter({k: v for k, v in counts_all.items() if k not in EXTRA_STOP})     # 処理後 / after
print(counts_all.most_common(20))
print(counts.most_common(12))

### 頻出語 / Most frequent words (Figure 3-2-5)

図3-2-5は、除外語を追加する前の集計です。ワードクラウドは追加後の集計で作ります。
Figure 3-2-5 shows the counts before adding extra stop words. The word cloud uses the counts after removing them.

In [ ]:
top = counts_all.most_common(20)[::-1]
fig, ax = plt.subplots(figsize=(FIG_W, 3.4))
ax.barh([w for w, _ in top], [c for _, c in top], color="#777777", edgecolor="black", lw=0.5)
for i, (_, c) in enumerate(top):
    ax.text(c + 0.5, i, str(c), va="center", fontsize=6.5)
ax.set_xlabel(L["count"]); ax.tick_params(axis="y", labelsize=7)
fig.tight_layout(); save(fig, "fig3-2-5_word_freq"); plt.show()

### ワードクラウド / Word cloud (Figure 3-2-6)

In [ ]:
from wordcloud import WordCloud

def gray(word, font_size, **kwargs):
    return f"hsl(0,0%,{max(0, 45 - font_size * 0.35):.0f}%)"   # 黒〜濃いグレー / black to dark gray

wc = WordCloud(font_path=JP_FONT if LANG == "ja" else None, width=1350, height=700,
               background_color="white", color_func=gray, prefer_horizontal=1.0, random_state=1,
               max_font_size=130, margin=22, relative_scaling=0.4, max_words=35).generate_from_frequencies(counts)
fig, ax = plt.subplots(figsize=(FIG_W, 2.35))
ax.imshow(wc); ax.axis("off")
fig.tight_layout(pad=0); save(fig, "fig3-2-6_wordcloud"); plt.show()

### 元の文章に戻る / Going back to the original comments

In [ ]:
keyword = "送料" if LANG == "ja" else "shipping"
for c in comments[comments.str.contains(keyword, case=False)].head(10):
    print("・", c)

### 価値観と要望を結びつける（発展） / Linking values and requests (advanced)

In [ ]:
price = F.iloc[:, 0]
hi = price >= price.quantile(0.75)
lo = price <= price.quantile(0.25)
for label, mask in [("価格重視・上位25% / Top 25%", hi), ("価格重視・下位25% / Bottom 25%", lo)]:
    c = Counter(w for t in comments[mask] for w in words(t) if w not in STOP)
    print(label, c.most_common(10))

## 3-2-5 まとめ / Summary

満足度と因子得点の相関を確認します。 / Check the correlation between satisfaction and factor scores.

In [ ]:
print(F.corrwith(df["satisfaction"]).round(2))